# COS30019 - Assignment 2 Part B
## Data Processing for Traffic Flow Prediction

### Purpose

This notebook documents my data-processing contribution for the Traffic-Based Route Guidance System (TBRGS). The supplied SCATS dataset records daily traffic volumes for Boroondara locations using 96 values per day, with each value representing a 15-minute interval. Before the machine-learning models can use this data, it must be checked and transformed into a reliable time-series structure.

The pipeline below performs the following tasks:

1. Loads the supplied SCATS Excel dataset.
2. Checks for missing, invalid, negative and duplicated traffic records.
3. Converts the original daily format (`V00` to `V95`) into timestamp-based traffic records.
4. **Detects and resolves duplicate site-location-timestamp records** introduced during reshaping by averaging the affected traffic-flow values.
5. Splits the dataset chronologically into training, validation and testing portions.
6. Applies min-max normalisation using training data only, which prevents information leakage.
7. Demonstrates sequence creation using the previous 12 intervals (3 hours) to predict the next traffic-flow value.

**Notes on behaviour:**
- Validation and test splits may contain traffic-flow values that exceed the training maximum, producing scaled values above 1.0. This is expected and correct — it is not a data-leakage issue.
- The sliding window in step 7 does not restart at midnight. It operates continuously over each (split, site, location) group, so the first window of a new calendar day includes observations from the previous day. This is intentional for continuous time-series modelling.
- The sequence window size is configurable. The final value should be agreed with the group so that LSTM, GRU and Transformer all use the same prepared input structure.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATASET_PATH = Path('../Scats Data October 2006.xls')
WINDOW_SIZE = 12  # 12 intervals = previous 3 hours
traffic_df = pd.read_excel(DATASET_PATH, sheet_name='Data', header=1, dtype={'SCATS Number': str})
traffic_df['SCATS Number'] = traffic_df['SCATS Number'].str.zfill(4)
traffic_df['Date'] = pd.to_datetime(traffic_df['Date'])
volume_columns = [f'V{i:02d}' for i in range(96)]
traffic_df.head()


### 1. Initial Data Quality Check
Before reshaping the dataset, the traffic fields are inspected for missing values, invalid entries, negative values and duplicated daily records. A second duplicate check is performed after reshaping to catch site-location-timestamp collisions that may arise from source rows that are not exact row-level duplicates.


In [ ]:
numeric_volume = traffic_df[volume_columns].apply(pd.to_numeric, errors='coerce')
missing_values = int(traffic_df[volume_columns].isna().sum().sum())
invalid_values = int(numeric_volume.isna().sum().sum() - missing_values)
negative_values = int((numeric_volume < 0).sum().sum())
duplicate_rows = int(traffic_df.duplicated().sum())
traffic_df[volume_columns] = numeric_volume

quality_summary = pd.DataFrame({
    'Check Performed': [
        'Daily records loaded', 'Unique SCATS sites', 'Location/direction entries',
        'Traffic columns', 'Missing traffic values', 'Invalid traffic values',
        'Negative traffic values', 'Duplicate daily records (exact row)'
    ],
    'Result': [
        len(traffic_df), traffic_df['SCATS Number'].nunique(),
        traffic_df[['SCATS Number','Location']].drop_duplicates().shape[0],
        len(volume_columns), missing_values, invalid_values,
        negative_values, duplicate_rows
    ]
})
quality_summary


### 2. Convert the Daily Table into a Time-Series Table
Each original row contains a full day of traffic readings. The following step converts each 15-minute reading into an individual timestamped record, which is more suitable for sequential prediction models.

After reshaping, a second duplicate check is performed on the `(scats_number, location, timestamp)` key. Any duplicates that arise from source rows sharing the same SCATS Number, Location and Date (but that are not identical rows) are resolved by averaging their traffic-flow values, ensuring each timestamp is unique per site and location.


In [ ]:
time_labels = {f'V{i:02d}': f'{i // 4:02d}:{(i % 4) * 15:02d}' for i in range(96)}
long_df = traffic_df.melt(
    id_vars=['SCATS Number','Location','NB_LATITUDE','NB_LONGITUDE','Date'],
    value_vars=volume_columns,
    var_name='time_slot', value_name='traffic_flow'
)
long_df['time'] = long_df['time_slot'].map(time_labels)
long_df['timestamp'] = pd.to_datetime(long_df['Date'].dt.strftime('%Y-%m-%d') + ' ' + long_df['time'])
long_df = long_df.rename(columns={'SCATS Number':'scats_number','Location':'location','NB_LATITUDE':'latitude','NB_LONGITUDE':'longitude','Date':'date'})
long_df = long_df[['scats_number','location','latitude','longitude','date','time_slot','time','timestamp','traffic_flow']].sort_values(['scats_number','location','timestamp']).reset_index(drop=True)

# Detect and resolve duplicate (scats_number, location, timestamp) records.
dup_key = ['scats_number', 'location', 'timestamp']
n_duplicates = int(long_df.duplicated(subset=dup_key).sum())
print(f'Duplicate site-location-timestamp records found: {n_duplicates}')
if n_duplicates > 0:
    meta_cols = ['scats_number','location','latitude','longitude','date','time_slot','time','timestamp']
    long_df = (
        long_df
        .groupby(dup_key, sort=False)
        .agg({**{c: 'first' for c in meta_cols if c not in dup_key}, 'traffic_flow': 'mean'})
        .reset_index()
    )[['scats_number','location','latitude','longitude','date','time_slot','time','timestamp','traffic_flow']]
    long_df = long_df.sort_values(['scats_number','location','timestamp']).reset_index(drop=True)
    print(f'Resolved by averaging. Remaining records: {len(long_df)}')

print('Time-series records created:', len(long_df))
long_df.head()


### 3. Chronological Split and Normalisation
The split is kept chronological because future traffic observations should not be used to train a model that predicts earlier observations. Min-max normalisation is fitted on training data only to avoid data leakage.

**Note on scaled values above 1.0:** Validation and test splits may contain traffic-flow peaks that exceed the training-set maximum (636). This causes some scaled values to exceed 1.0. This is expected behaviour — not a bug or leakage. The model must tolerate inputs slightly outside the [0, 1] training range.


In [ ]:
dates = sorted(long_df['date'].dt.date.unique())
train_dates, validation_dates, test_dates = dates[:21], dates[21:26], dates[26:]
long_df['dataset_split'] = np.select([
    long_df['date'].dt.date.isin(train_dates),
    long_df['date'].dt.date.isin(validation_dates),
    long_df['date'].dt.date.isin(test_dates)
], ['train','validation','test'], default='unassigned')
train_min = float(long_df.loc[long_df['dataset_split']=='train','traffic_flow'].min())
train_max = float(long_df.loc[long_df['dataset_split']=='train','traffic_flow'].max())
long_df['traffic_flow_scaled'] = (long_df['traffic_flow'] - train_min) / (train_max - train_min)

# Warn about any split with scaled values above 1.0.
for split in ['validation', 'test']:
    split_max = long_df.loc[long_df['dataset_split']==split,'traffic_flow_scaled'].max()
    if split_max > 1.0:
        print(f"Note: '{split}' max scaled value = {split_max:.4f} (above 1.0). "
              f"Expected — its traffic peak exceeds the training maximum of {train_max}.")

split_summary = long_df.groupby('dataset_split', as_index=False).agg(
    first_date=('date','min'), last_date=('date','max'),
    records=('traffic_flow','size'), minimum_flow=('traffic_flow','min'),
    maximum_flow=('traffic_flow','max')
)
# Format dates as date-only (not datetime) for clean display.
split_summary['first_date'] = split_summary['first_date'].dt.date
split_summary['last_date'] = split_summary['last_date'].dt.date
split_summary


### 4. Configurable Time-Series Window Creation
For demonstration, the previous 12 observations (3 hours) are used to predict the next 15-minute traffic value. This value can later be changed to match the final model settings agreed by the team.

**Note on cross-midnight windows:** The sliding window does not restart at midnight. It operates continuously over each (split, site, location) group, so the first window of a new calendar day includes observations from the previous day. This is intentional — traffic flow is a continuous signal and artificially breaking the sequence at midnight would discard valid recent context.


In [ ]:
def create_sequences(df, window_size=WINDOW_SIZE):
    X, y, details = [], [], []
    for (split, site, location), group in df.groupby(['dataset_split','scats_number','location'], sort=False):
        group = group.sort_values('timestamp').reset_index(drop=True)
        values = group['traffic_flow_scaled'].to_numpy(dtype=np.float32)
        for i in range(window_size, len(values)):
            X.append(values[i-window_size:i])
            y.append(values[i])
            details.append({'dataset_split': split, 'scats_number': site, 'location': location, 'target_timestamp': group.loc[i,'timestamp'], 'actual_traffic_flow': group.loc[i,'traffic_flow']})
    return np.asarray(X), np.asarray(y), pd.DataFrame(details)

X_all, y_all, sequence_metadata = create_sequences(long_df)
for split in ['train','validation','test']:
    mask = sequence_metadata['dataset_split'] == split
    print(split, 'X shape:', X_all[mask.to_numpy()].shape, 'y shape:', y_all[mask.to_numpy()].shape)


### Processing Outcome
The source dataset was successfully validated and converted into timestamp-based traffic records. No missing, invalid or negative traffic-flow values were identified. The prepared output provides a chronological and leakage-aware foundation that can be shared with the model-development members for consistent training and evaluation.
